# Comparing models to fMRI data using BayesCompare
This tutorial shows how to use the `encoding` module's functions in BayesCompare in order to compare models to brain data. We will focus on how to prepare the brain data, extract the model's representations, and finally compute the comparisons using BayesCompare.

## Obtaining the dataset
In order to download the data, please fill the [NSD Data Access Agreement Form](https://docs.google.com/forms/d/e/1FAIpQLSduTPeZo54uEMKD-ihXmRhx0hBDdLHNsVyeo_kCb8qbyAkXuQ/viewform). You will then receive an email with a link to the documentation and instructions on how to access the data. For this tutorial, we will use the [NSD Synthetic dataset](https://cvnlab.slite.page/p/wLlJyfWRvg/Functional-data-nsdsynthetic). After filling out the form, we can download the data using `aws-cli`

In [2]:
%%bash
mkdir encoding_data
aws s3 sync --no-sign-request s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic encoding_data/stimuli
aws s3 sync --no-sign-request s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic_subj01 encoding_data/stimuli
aws s3 sync --no-sign-request s3://natural-scenes-dataset/nsddata/experiments/nsdsynthetic encoding_data/nsdsynthetic
for i in {1..8}; do
  aws s3 cp --no-sign-request s3://natural-scenes-dataset/nsddata_betas/ppdata/subj0${i}/func1pt8mm/nsdsyntheticbetas_fithrf/betas_nsdsynthetic.hdf5 encoding_data/data/subj0${i}/betas_nsdsynthetic.hdf5;
  aws s3 cp --no-sign-request s3://natural-scenes-dataset/nsddata/ppdata/subj0${i}/func1pt8mm/roi/prf-visualrois.nii.gz encoding_data/rois/subj0${i}/prf-visualrois.nii.gz;
done

mkdir: cannot create directory ‘paper_example’: File exists


download: s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic/nsdsynthetic008.png to paper_example/stimuli/nsdsynthetic008.png
download: s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic/nsdsynthetic005.png to paper_example/stimuli/nsdsynthetic005.png
download: s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic/nsdsynthetic007.png to paper_example/stimuli/nsdsynthetic007.png
download: s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic/nsdsynthetic006.png to paper_example/stimuli/nsdsynthetic006.png
download: s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic/nsdsynthetic014.png to paper_example/stimuli/nsdsynthetic014.png
download: s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic/nsdsynthetic010.png to paper_example/stimuli/nsdsynthetic010.png
download: s3://natural-scenes-dataset/nsddata/stimuli/nsdsynthetic/nsdsynthetic/nsdsynthetic002.png to paper_example/stimuli/nsdsy

## Imports

In [112]:
import h5py
import os

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import scipy.io as sio

from scipy.stats import zscore

## Preparing fMRI data
First, we need to get voxel data from the NSD files. For that, we will load the voxels corresponding to a single ROI (V1) in memory. The data is available in hdf5 format, which allows us to only load the voxels we select in memory, instead of the whole array. We will load the V1 ROI definitions using `prf-visualrois.nii.gz`. In order to use a different visual ROI, you can change the index used for the voxel selection. Additional ROIs can be selected using the same logic shown below (follow the documentation [here](https://cvnlab.slite.page/p/X_7BBMgghj/ROIs) for other ROIs).

In [113]:
# Define data path and sub list
data_dir = 'encoding_data/data'
subjects = [f"subj0{i}" for i in range(1, 9)]  # [subj01, ..., subj08]

betas_fnames = [os.path.join(data_dir, sub, "betas_nsdsynthetic.hdf5") for sub in subjects]

# Load V1 ROI mask from NSD data
#for idx, sub in enumerate(subjects):
sub_fname = betas_fnames[0]
sub = subjects[0]
print(f"Loading ROI for {sub}...")
roi_path = f'encoding_data/rois/{sub}/prf-visualrois.nii.gz'  # Load a different mask to use other ROIs
roi_img = nib.load(roi_path)
roi_data = roi_img.get_fdata()

# V1 is labeled as 1 in prf-visualrois -- change the number to use other visual ROI
v1_mask = (roi_data == 1).T  # Need transpose to match betas dimensions
target_idx = np.where(v1_mask)
print(f"V1 voxels: {v1_mask.sum()}")

# Get betas
with h5py.File(sub_fname, 'r') as f:
    sub_betas = np.stack([f['betas'][..., idx[0], idx[1], idx[2]] for idx in zip(*target_idx)]).astype(float) / 300
print(f"Shape of V1 betas for {sub}: {sub_betas.shape}")  # should be (num_voxels_in_V1, 744)

Loading ROI for subj01...
V1 voxels: 594
Shape of V1 betas for subj01: (594, 744)


In [114]:
image_info.loc[master_ordering]

,Image number,Image,Image subclass number,Image subclass,Image class number,Image class
197,198,spiral_D_sf6_2,50,SpiralD SF6,7,Spiral gratings
230,231,hue03_3,58,Hue03,8,Chromatic noise
195,196,spiral_D_sf5_4,49,SpiralD SF5,7,Spiral gratings
151,152,spiral_B_sf6_4,38,SpiralB SF6,7,Spiral gratings
52,53,phase050_1,14,Phase 50%,5,Phase-coherence modulation
...,...,...,...,...,...,...
18,19,upsidedown_3,5,Upside-down scenes,3,Manipulated scenes
20,21,mooney_1,6,Mooney scenes,3,Manipulated scenes
6,7,whitenoiseLB_3,2,White noise (large block),1,Noise
12,13,natscene_1,4,Natural scenes,2,Natural scenes


## Obtaining image IDs
The second thing we need is to obtain information about the images shown to the subjects (the first dimension in our betas). We only need to know the image number corresponding to each image, which we can obtain from the `nsdsynthetic_expdesign.mat` file. Finally, we can make a dataframe where the rows are our voxels and the columns are the image numbers.

In [115]:
# Load NSD synthetic experimental design
expdesign_path = 'encoding_data/nsdsynthetic/nsdsynthetic_expdesign.mat'
expdesign = sio.loadmat(expdesign_path)

# master_ordering maps the final row index to rows in image_info
master_ordering = expdesign['masterordering'][0]  # shape (744,)

# Fix 1-indexing from MATLAB to 0-indexing in Python
master_ordering = master_ordering - 1

# Make DataFrame containing image number
beta_df = pd.DataFrame(sub_betas, columns=master_ordering)

The reason we create this dataframe is to have a convenient way to re-arrange the columns so repetitions of the same image are grouped, which is necessary for later steps:

In [116]:
# Group columns by image
unique_imgs = beta_df.columns.unique()
beta_df = beta_df.loc[:, unique_imgs]  # Columns corresponding to the same image are grouped together
print(beta_df.head())

         197        197        230       230        195       195        151  \
0  24.146667 -14.536667  19.020000 -1.480000  21.413333 -5.223333  32.186667   
1  -3.590000  -7.700000   0.963333 -2.450000   3.623333 -8.330000   4.503333   
2  -7.293333  -0.220000  -7.683333  3.516667  -1.790000  1.540000   0.996667   
3  20.123333  -3.573333  13.460000  2.966667  12.616667  3.193333  23.473333   
4   0.960000  -0.593333  -2.450000  3.710000   0.943333  3.106667   4.416667   

        151        52        52   ...        211        211        266  \
0 -8.143333  21.683333  6.730000  ...   5.500000  12.420000   4.976667   
1  3.230000   4.440000  6.670000  ...  10.346667   5.103333  10.960000   
2  2.493333   5.303333  8.256667  ...  -1.370000   0.740000  -1.220000   
3 -1.013333  15.763333  9.640000  ...   2.166667   9.270000   1.706667   
4  2.153333   1.246667  9.903333  ...   4.320000   3.320000   3.650000   

        266       222       222        222       222        153        153

## Obtaining noise estimates and mean activations
The BayesCompare method requires an estimation of the noise present in the brain data in order to compute the similarity with different models. We can estimate the noise in the data using the `voxel_reliability` function provided by BayesCompare. To make things easier, we will normalize the data so the total variance of each voxel is equal to 1.

In [129]:
from BayesCompare.encoding_utils import voxel_reliability

# Normalize data - we keep it in DataFrame shape in order to get the mean activations per image later
norm_betas = pd.DataFrame(
    zscore(beta_df.values, axis=1),
    beta_df.index,
    beta_df.columns
)

# Obtain reliability, noise variance and total variance estimates from the data
voxel_data = norm_betas.values
stim_list = norm_betas.columns.values
reliability, noise_var, total_var = voxel_reliability(voxel_data, stim_list)

# We can check that the total variance for each voxel is close to 1
print(f"Number of voxels: {total_var.shape[0]}\nTotal variance across all voxels: {total_var.sum()}")

Number of voxels: 594
Total variance across all voxels: 594.7994616419919


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.005471706390380859s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.009648799896240234s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done  50 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done  76 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 106 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.010477542877197266s.) Setting batch_size=8.
[Parallel(n_jobs=-1)]: Done 152 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 220 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.011652469635009766s.)